# Colab Acceleration: Qwen2.5-VL Descriptions & Google SigLIP Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xenodion/deep-learning-project-waerniers-lamy/blob/feat/nlp-recommendation/src/deep_learning_project_waerniers_lamy/nlp/Colab_VLM_and_SigLIP_Enrichment.ipynb)

This Google Colab notebook provides GPU-accelerated batch extraction for:
1. **High-Definition VLM Descriptions (`Qwen/Qwen2.5-VL-3B-Instruct`)** generated in safe micro-batches of 100 images with automatic incremental downloads.
2. **State-of-the-Art Multimodal Embeddings (`google/siglip-base-patch16-224`)**.
3. **Audited, 100% Reliable Benchmark Evaluation** on active corpus images (eliminating artificial 0% penalties for out-of-corpus targets).
4. **Hybrid Multimodal Retrieval** (SigLIP visual similarity + Qwen semantic text).

**Hardware requirement:** Google Colab Free T4 GPU (Runtime > Change runtime type > T4 GPU).

## 1. Setup Environment & Fix Dependencies
Installs modern Hugging Face packages, sentencepiece, and applies PyTorch / NumPy compatibility patches.

In [ ]:
# 1. Install required packages (including sentencepiece for SigLIP and scipy upgrade)
!pip install -q --upgrade scipy sentencepiece protobuf transformers accelerate torchvision pillow datasets sentence-transformers peft qwen-vl-utils

# Core imports & NumPy / PyTorch compatibility patches
import os
import sys
import json
import time
from pathlib import Path
import numpy as np

# FIX: Patch numpy 2.x missing symbol required by scipy
if hasattr(np, '_core') and hasattr(np._core, '_multiarray_umath'):
    if not hasattr(np._core._multiarray_umath, '_blas_supports_fpe'):
        setattr(np._core._multiarray_umath, '_blas_supports_fpe', lambda *args, **kwargs: False)

import torch
import torch._utils  # Prevents 'module torch has no attribute _utils' bug in Colab
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from google.colab import files

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > Select T4 GPU.")


## 2. Setup Repository & Dataset
Clones the codebase, sets up `sys.path`, and restores your dataset (from uploaded zip or Hugging Face).

In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path

# 1. Démarrer depuis /content pour éviter l'imbrication repo/repo
%cd /content

repo_dir = Path("/content/repo")
if not (repo_dir / ".git").exists():
    !git clone -b feat/nlp-recommendation https://github.com/Xenodion/deep-learning-project-waerniers-lamy.git /content/repo
else:
    !cd /content/repo && git pull origin feat/nlp-recommendation

# 2. Installer le repo en mode éditable et injecter src dans sys.path
!pip install -q -e /content/repo
src_path = str(repo_dir / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

%cd /content/repo

# 3. Vérifier la présence du dataset (upload zip ou streaming HuggingFace)
zip_candidates = [Path("/content/wikiart_genre5.zip"), Path("/content/repo/wikiart_genre5.zip")]
uploaded_zip = next((p for p in zip_candidates if p.exists()), None)
data_target = repo_dir / "data" / "wikiart_genre5"
data_target.parent.mkdir(parents=True, exist_ok=True)

if not (data_target / "metadata.csv").exists():
    if uploaded_zip:
        print(f"Extraction du zip uploadé : {uploaded_zip}...")
        extract_tmp = Path("/content/tmp_extract")
        with zipfile.ZipFile(uploaded_zip, "r") as z:
            z.extractall(extract_tmp)
        found_dir = None
        for root, dirs, files_in_dir in os.walk(extract_tmp):
            if "metadata.csv" in files_in_dir:
                found_dir = Path(root)
                break
        if found_dir:
            if data_target.exists():
                shutil.rmtree(data_target)
            shutil.move(str(found_dir), str(data_target))
            print(f"Données installées avec succès dans {data_target} !")
    else:
        print("Téléchargement du subset WikiArt 5 genres (~1-2 min)...\n")
        !python src/deep_learning_project_waerniers_lamy/vision/make_wikiart_subset.py \
            --label genre \
            --classes portrait landscape cityscape still_life religious_painting \
            --per-class 600 --size 224 --out data/wikiart_genre5
else:
    print(f"Dataset (3000 images) déjà présent et validé dans {data_target} !")

# 4. Récupération des descriptions déjà existantes si uploadées dans /content
for fname in ["descriptions_qwen_val.json", "descriptions_qwen_all.json", "descriptions_val.json"]:
    uploaded_f = Path("/content") / fname
    if uploaded_f.exists() and not (data_target / fname).exists():
        shutil.copy(uploaded_f, data_target / fname)
        print(f"Fichier de descriptions existant restauré : {fname}")

# 5. Générer le split Museum A / B directement en Python si nécessaire
if not (data_target / "split.csv").exists():
    from deep_learning_project_waerniers_lamy.vision.utils import make_split
    make_split(data_dir=data_target)
    print("Split Museum A / B généré avec succès !")
else:
    print("split.csv vérifié et prêt.")


## 3. Part A: Load Qwen2.5-VL-3B Vision-Language Model
> **Architecture Decision (ADR 0005 — Length Calibration):**
> The prompt is strictly constrained to **2 sentences (~35–45 words)**. This strikes the optimal balance between rich visual vocabulary (subjects, palette, lighting) and preventing semantic dilution during mean-pooling.

In [ ]:
# Fix de compatibilité NumPy 2.x / SciPy pour AutoProcessor
import numpy as np
if hasattr(np, '_core') and hasattr(np._core, '_multiarray_umath'):
    if not hasattr(np._core._multiarray_umath, '_blas_supports_fpe'):
        setattr(np._core._multiarray_umath, '_blas_supports_fpe', lambda *args, **kwargs: False)

import torch
import torch._utils
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

vlm_model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
print(f"Loading {vlm_model_name} on GPU (float16)...")

processor = AutoProcessor.from_pretrained(vlm_model_name)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    vlm_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Qwen2.5-VL-3B loaded successfully on GPU!")


## 4. Batch Generation in Micro-Chunks of 100 with Auto-Download
Generates descriptions in safe batches of **100 images (~1.5 to 2 minutes per chunk)**.
At the end of each chunk, the progress is saved and **automatically downloaded** to your computer to prevent losing work if Colab disconnects.

In [ ]:
from deep_learning_project_waerniers_lamy.nlp.utils import load_paintings

# Choix du split à générer : 'val' (480 images), 'train' (1920 images), ou ['train', 'val'] (2400 images)
TARGET_SPLIT = "val"  # Changez en 'train' ou ['train', 'val'] pour le reste de Museum A
CHUNK_SIZE = 100       # Tranches de 100 images (~1.5 à 2 min par tranche)

data_dir = Path("/content/repo/data/wikiart_genre5")
paintings = load_paintings(data_dir=data_dir, split=TARGET_SPLIT)
print(f"Corpus cible : {len(paintings)} tableaux (split={TARGET_SPLIT}).")

PROMPT = (
    "Analyze this fine-art painting concisely for museum visitors. "
    "Describe the visible subjects, setting, landscape, weather, colors and atmosphere in 2 sentences."
)

# Charger la base globale existante pour ne jamais ré-écraser
output_global_path = data_dir / f"descriptions_qwen_{TARGET_SPLIT if isinstance(TARGET_SPLIT, str) else 'all'}.json"
all_descriptions = {}

# Reprise automatique
for candidate in [output_global_path, Path("/content/descriptions_qwen_val.json"), data_dir / "descriptions_qwen_val.json"]:
    if candidate.exists():
        try:
            d = json.loads(candidate.read_text(encoding="utf-8"))
            all_descriptions.update(d)
            print(f"Reprise de {len(d)} descriptions depuis {candidate.name}.")
        except Exception:
            pass

# Identifier les tableaux restant à générer
remaining_paintings = paintings[~paintings["id"].isin(all_descriptions)].reset_index(drop=True)
print(f"Total déjà décrits : {len(all_descriptions)} | Restants à générer : {len(remaining_paintings)}/{len(paintings)}")

total_remaining = len(remaining_paintings)
for chunk_start in range(0, total_remaining, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_remaining)
    chunk_num = (chunk_start // CHUNK_SIZE) + 1
    chunk_df = remaining_paintings.iloc[chunk_start:chunk_end]
    
    print(f"\n▶ Démarrage Tranche {chunk_num} : {chunk_start} à {chunk_end} / {total_remaining}...")
    
    chunk_dict = {}
    for idx, (_, row) in enumerate(tqdm(chunk_df.iterrows(), total=len(chunk_df))):
        p_id = row["id"]
        img_path = str(data_dir / row["path"])
        if not os.path.exists(img_path):
            continue
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img_path},
                    {"type": "text", "text": PROMPT}
                ]
            }
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=60)
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            caption = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0].strip()
        
        all_descriptions[p_id] = caption
        chunk_dict[p_id] = caption
    
    # Sauvegarde globale sur disque Colab
    output_global_path.write_text(json.dumps(all_descriptions, indent=2, ensure_ascii=False), encoding="utf-8")
    
    # Sauvegarde et téléchargement immédiat du chunk de 100
    chunk_file = Path(f"/content/descriptions_chunk_{chunk_num}.json")
    chunk_file.write_text(json.dumps(chunk_dict, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n Tranche {chunk_num} terminée ({len(all_descriptions)} descriptions totales prêtes).")
    try:
        files.download(str(chunk_file))
        print(f" Téléchargement automatique lancé : {chunk_file.name}")
    except Exception as e:
        print(f"Note : fichier sauvé dans {chunk_file}")

print(f"\n Session de génération terminée ! Total disponible : {len(all_descriptions)} descriptions.")


## 5. Part B: State-of-the-Art Multimodal Embeddings with Google SigLIP

In [ ]:
from transformers import AutoModel, AutoProcessor

siglip_name = "google/siglip-base-patch16-224"
print(f"Loading SigLIP model {siglip_name} on GPU...")

siglip_processor = AutoProcessor.from_pretrained(siglip_name)
siglip_model = AutoModel.from_pretrained(siglip_name, torch_dtype=torch.float16).cuda()
siglip_model.eval()
print("SigLIP loaded successfully on GPU!")


In [ ]:
# Batch encode validation paintings with SigLIP (~20 seconds on T4 GPU)
data_dir = Path("/content/repo/data/wikiart_genre5")
paintings_val = load_paintings(data_dir=data_dir, split="val")
batch_size = 32
all_siglip_embeddings = []
all_siglip_ids = []

print(f"Encoding {len(paintings_val)} validation images with SigLIP...")
for start_idx in range(0, len(paintings_val), batch_size):
    batch_rows = paintings_val.iloc[start_idx : start_idx + batch_size]
    images = [Image.open(data_dir / row["path"]).convert("RGB") for _, row in batch_rows.iterrows()]
    valid_ids = batch_rows["id"].tolist()

    inputs = siglip_processor(images=images, return_tensors="pt").to("cuda", dtype=torch.float16)
    with torch.no_grad():
        img_feats = siglip_model.get_image_features(**inputs)
        if hasattr(img_feats, "pooler_output") and img_feats.pooler_output is not None:
            img_feats = img_feats.pooler_output
        elif not isinstance(img_feats, torch.Tensor):
            img_feats = img_feats[0]
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        all_siglip_embeddings.append(img_feats.cpu().float().numpy())
        all_siglip_ids.extend(valid_ids)

siglip_matrix = np.vstack(all_siglip_embeddings)
siglip_emb_path = Path("/content/repo/siglip_embeddings_val.npy")
siglip_ids_path = Path("/content/repo/siglip_ids_val.json")

np.save(siglip_emb_path, siglip_matrix)
siglip_ids_path.write_text(json.dumps(all_siglip_ids), encoding="utf-8")
print(f"Saved SigLIP matrix {siglip_matrix.shape} to {siglip_emb_path}!")


## 6. Part C: 100% Reliable & Audited Benchmark Evaluation
Evaluates only queries that have at least one valid target painting in the active collection, eliminating artificial penalties for images outside the corpus.

In [ ]:
from sentence_transformers import SentenceTransformer
from deep_learning_project_waerniers_lamy.nlp.benchmark import get_benchmark_queries
from deep_learning_project_waerniers_lamy.nlp.utils import evaluate_retrieval_benchmark

# 1. Chargement des descriptions et vectorisation texte
qwen_dict = {}
for p_cand in [data_dir / "descriptions_qwen_val.json", data_dir / "descriptions_qwen_all.json", Path("/content/descriptions_qwen_val.json")]:
    if p_cand.exists():
        qwen_dict.update(json.loads(p_cand.read_text(encoding="utf-8")))

paintings_val["description"] = paintings_val["id"].map(qwen_dict).fillna("")
texts = [
    f"{r['description']}. {r['genre']} style {r['style']}" if r["description"] else f"{r['genre']} style {r['style']}"
    for _, r in paintings_val.iterrows()
]

print("Encoding descriptions with all-MiniLM-L6-v2...")
text_embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = text_embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
painting_ids = paintings_val["id"].tolist()
siglip_tensor = torch.tensor(siglip_matrix, dtype=torch.float32).cuda()

# 2. Benchmark audité : filtre autonome ne dépendant pas d'un git push
raw_queries = get_benchmark_queries()
val_set = set(painting_ids)
eval_benchmark = []
for q in raw_queries:
    targets = [x for x in q.get('relevant_ids', []) if x in val_set]
    if targets:
        q_copy = dict(q)
        q_copy['relevant_ids'] = targets
        eval_benchmark.append(q_copy)

print(f"Benchmark audité : {len(eval_benchmark)} requêtes valides (avec au moins 1 cible dans la collection active).")

# 3. Fonctions de recherche
def retrieve_siglip(query: str, top_k: int = 10):
    inputs = siglip_processor(text=[query], return_tensors="pt", padding="max_length").to("cuda")
    with torch.no_grad():
        t_feats = siglip_model.get_text_features(**inputs)
        if hasattr(t_feats, "pooler_output") and t_feats.pooler_output is not None:
            t_feats = t_feats.pooler_output
        elif not isinstance(t_feats, torch.Tensor):
            t_feats = t_feats[0]
        t_feats = t_feats.to(dtype=siglip_tensor.dtype)
        t_feats = t_feats / t_feats.norm(dim=-1, keepdim=True)
    scores = torch.matmul(siglip_tensor, t_feats.T).squeeze(-1).cpu().numpy()
    return [painting_ids[i] for i in np.argsort(scores)[::-1][:top_k]]

def retrieve_qwen(query: str, top_k: int = 10):
    q_emb = text_embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = torch.matmul(doc_embeddings, q_emb).cpu().numpy()
    return [painting_ids[i] for i in np.argsort(scores)[::-1][:top_k]]

def retrieve_hybrid(query: str, top_k: int = 10, alpha: float = 0.5):
    inputs = siglip_processor(text=[query], return_tensors="pt", padding="max_length").to("cuda")
    with torch.no_grad():
        t_feats = siglip_model.get_text_features(**inputs)
        if hasattr(t_feats, "pooler_output") and t_feats.pooler_output is not None:
            t_feats = t_feats.pooler_output
        elif not isinstance(t_feats, torch.Tensor):
            t_feats = t_feats[0]
        t_feats = t_feats.to(dtype=siglip_tensor.dtype)
        t_feats = t_feats / t_feats.norm(dim=-1, keepdim=True)
    score_img = torch.matmul(siglip_tensor, t_feats.T).squeeze(-1).cpu().numpy()
    
    q_emb = text_embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    score_txt = torch.matmul(doc_embeddings, q_emb).cpu().numpy()
    
    score_comb = alpha * score_img + (1.0 - alpha) * score_txt
    return [painting_ids[i] for i in np.argsort(score_comb)[::-1][:top_k]]

# 4. Évaluations comparatives
m_sig = evaluate_retrieval_benchmark(retrieve_siglip, eval_benchmark, k_values=(5, 10))
m_qwen = evaluate_retrieval_benchmark(retrieve_qwen, eval_benchmark, k_values=(5, 10))
m_hyb = evaluate_retrieval_benchmark(lambda q, k: retrieve_hybrid(q, k, alpha=0.5), eval_benchmark, k_values=(5, 10))

print("=" * 70)
print("RÉSULTATS AUDITÉS SUR LA COLLECTION VALIDÉE")
print(f"1. Google SigLIP seul  -> Recall@5: {m_sig['recall_at_5']*100:.1f}% | Recall@10: {m_sig['recall_at_10']*100:.1f}% | MRR: {m_sig['mrr']:.3f}")
print(f"2. Qwen2.5-VL seul     -> Recall@5: {m_qwen['recall_at_5']*100:.1f}% | Recall@10: {m_qwen['recall_at_10']*100:.1f}% | MRR: {m_qwen['mrr']:.3f}")
print(f"3. HYBRIDE (50/50)     -> Recall@5: {m_hyb['recall_at_5']*100:.1f}% | Recall@10: {m_hyb['recall_at_10']*100:.1f}% | MRR: {m_hyb['mrr']:.3f}")
print("=" * 70)


## 7. Part D: Export & Download Generated Artifacts
Zips all generated files (`descriptions_qwen_*.json`, `siglip_embeddings_val.npy`, `siglip_ids_val.json`) and downloads the final archive.

In [ ]:
from google.colab import files

# Archive all generated artifacts into one zip
!zip -j -q /content/nlp_boost_artifacts.zip \
    /content/repo/data/wikiart_genre5/descriptions_qwen_*.json \
    /content/repo/siglip_embeddings_val.npy \
    /content/repo/siglip_ids_val.json \
    /content/descriptions_chunk_*.json

print("Téléchargement de /content/nlp_boost_artifacts.zip...")
files.download('/content/nlp_boost_artifacts.zip')
print("Téléchargement lancé avec succès !")
